In [25]:
from user_api import *

In [26]:
import pandas as pd

try:
    df = pd.read_parquet("../data/raw/games.parquet")
    print("Cargado desde Parquet.")
except Exception:
    df = pd.read_pickle("data/raw/games.pkl")
    print("Cargado desde Pickle.")

print(df.shape)
df

Cargado desde Parquet.
(122611, 43)


,app_id,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0,0.00,0,,,,,...,0,0,0 - 0,0,0,0,0,0,0,[]
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,5.24,0,"Springtime, April: when the cherry trees come ...","Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",,...,252,3,0 - 20000,8,0,8,0,65,0,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':..."
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0,4.99,0,"Immerse yourself in the most beloved, mystical...","Immerse yourself in the most beloved, mystical...",Discover an entrancing and spectacular world!,,...,21,3,0 - 20000,0,0,0,0,0,0,"{'Casual': 83, 'Card Game': 52, 'Solitaire': 4..."
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0,8.99,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...","synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",Yuha! I'll start the broadcast! Hakko's extrem...,,...,0,0,0 - 20000,0,0,0,0,0,1,[]
4,3631080,Maze Quest VR,"Apr 24, 2025",0,4.99,0,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,,...,0,0,0 - 20000,0,0,0,0,0,0,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122606,4152910,完美传奇,"Jan 4, 2026",0,0.00,0,《完美传奇》游戏介绍 🔥【完美传奇——专属大陆的奇幻史诗！】🔥 🌍耗时两年匠心打造，诚意之作...,《完美传奇》游戏介绍 🔥【完美传奇——专属大陆的奇幻史诗！】🔥 🌍耗时两年匠心打造，诚意之作...,欢迎来到《完美传奇》！ 🌍专属大陆×自由探索，六道轮回剧情×百变BUFF搭配，一刀爆神装、机...,,...,0,0,0 - 0,0,0,0,0,0,0,[]
122607,4042800,Poker Fate - ACG Texas Hold'em,"Jan 3, 2026",0,0.00,0,Poker Fate – Choose your poker partner and ent...,Poker Fate – Choose your poker partner and ent...,Poker Fate is an anime-style Texas Hold'em gam...,,...,0,0,0 - 0,0,0,0,0,0,0,[]
122608,3522550,Adira Nusantara,"Jan 3, 2026",0,7.99,0,(Gif character) Adira Nusantara is a side-scro...,(Gif character) Adira Nusantara is a side-scro...,"Master authentic Silat combat as Adira, a fier...",,...,0,0,0 - 0,0,0,0,0,0,0,[]
122609,3680350,A Lenda de Niterói,"Jan 4, 2026",0,2.09,0,"Step into the role of Arariboia, a legendary I...","Step into the role of Arariboia, a legendary I...",Embark on Arariboia’s journey during the 16th-...,,...,0,0,0 - 0,0,0,0,0,0,0,[]


In [27]:
COLUMNS = [
    "tags",
    "genres",
    "categories"
]

In [28]:
import numpy as np

def is_empty(value):
    if value is None:
        return True

    if isinstance(value, float) and np.isnan(value):
        return True

    if isinstance(value, (list, tuple)):
        return len(value) == 0

    if isinstance(value, np.ndarray):
        return value.size == 0

    if isinstance(value, str):
        return value.strip() == "" or value.strip() == "[]"

    return False

In [29]:
# Mask for completely empty rows
mask_all_empty = df.apply(
    lambda row: all(is_empty(row[col]) for col in COLUMNS),
    axis=1
)

# Filter those games
games_without_metadata = df[mask_all_empty]

print("📊 Juegos sin metadata en genres, tags y categories:")
print("Total:", len(games_without_metadata))

# Show only names (first 50)
print("\n🎮 Primeros juegos sin metadata:")
games_without_metadata[["app_id", "name", "tags", "genres", "categories"]].head(50)

📊 Juegos sin metadata en genres, tags y categories:
Total: 7984

🎮 Primeros juegos sin metadata:


,app_id,name,tags,genres,categories
0,2539430,Black Dragon Mage Playtest,[],[],[]
10,1946890,Codename: Warlock Playtest,[],[],[]
24,2349750,CyberVault Playtest,[],[],[]
25,2628280,A Night In Omar's Burger Playtest,[],[],[]
51,2689730,Dark and Deep Playtest,[],[],[]
97,2782210,Bill's Misfortune Playtest,[],[],[]
102,3664570,QUADRA CLASH Playtest,[],[],[]
116,3581240,Descendance Playtest,[],[],[]
118,1549730,Red Solstice 2: Survivors Playtest,[],[],[]
137,1639330,Pain Party Playtest,[],[],[]


# ESTO METER EN DATA CLEAN

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import ast

COLUMNS = ["genres", "tags", "categories"]

def clean_value(value):
    if value is None:
        return ""

    # NaN
    if isinstance(value, float) and np.isnan(value):
        return ""

    # ndarray
    if isinstance(value, np.ndarray):
        return " ".join(str(i).strip() for i in value if str(i).strip())

    # list
    if isinstance(value, list):
        return " ".join(str(i).strip() for i in value if str(i).strip())

    # dict
    if isinstance(value, dict):
        return " ".join(str(k).strip() for k in value.keys() if str(k).strip())

    # string
    if isinstance(value, str):
        text = value.strip()
        if text == "" or text == "[]":
            return ""

        # 🔥 If it looks like a list or dict, try parsing
        if text.startswith("[") or text.startswith("{"):
            try:
                parsed_value = ast.literal_eval(text)
                if isinstance(parsed_value, dict):
                    return " ".join(str(k).strip() for k in parsed_value.keys() if str(k).strip())
                if isinstance(parsed_value, list):
                    return " ".join(str(i).strip() for i in parsed_value if str(i).strip())
            except:
                pass  # if parsing fails, continue

        return text

    return ""


def remove_duplicates(text):
    words = text.split()
    return " ".join(dict.fromkeys(words))  # keeps order and removes duplicates


def prepare_text(df):
    df = df.copy()
    clean_columns = []

    for col in COLUMNS:
        if col in df.columns:
            series = (
                df[col]
                .map(clean_value)   # faster than apply
                .fillna("")
                .astype(str)
                .str.strip()
            )
            clean_columns.append(series)

    if not clean_columns:
        df["combined_features"] = ""
        return df

    # vectorized concatenation
    combined = clean_columns[0]
    for series in clean_columns[1:]:
        combined = combined.str.cat(series, sep=" ")

    # clean extra spaces
    df["combined_features"] = (
        combined
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .map(remove_duplicates)
    )

    return df


df = prepare_text(df)

print("📊 Filas con combined_features vacío:", (df["combined_features"] == "").sum())

# Opcional: ver cuáles son
print(df[df["combined_features"] == ""][["app_id", "name"]].head(20))

vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(df["combined_features"])

📊 Filas con combined_features vacío: 7984
      app_id                                         name
0    2539430                   Black Dragon Mage Playtest
10   1946890                   Codename: Warlock Playtest
24   2349750                          CyberVault Playtest
25   2628280            A Night In Omar's Burger Playtest
51   2689730                       Dark and Deep Playtest
97   2782210                   Bill's Misfortune Playtest
102  3664570                        QUADRA CLASH Playtest
116  3581240                         Descendance Playtest
118  1549730           Red Solstice 2: Survivors Playtest
137  1639330                          Pain Party Playtest
166  3392860                    Mechanical Pulse Playtest
186  3085750                                1971 Playtest
190  3973110                     Trickster Trove Playtest
236  1855530  From Warrior to Hero (Idle 3D RPG) Playtest
317  2205440                      Project OutFox Playtest
320  1761940                  

# PRUEBAS COMBINAR DATOS

In [ ]:
# 🔥 Eliminar juegos duplicados (mismo nombre)
df = df.sort_values("peak_ccu", ascending=False)
df = df.drop_duplicates(subset="name")

In [32]:
import re

def audit_combined_features(df):

    print("🔎 INICIANDO AUDITORÍA\n")

    total_rows = len(df)

    # 1️⃣ Tipo incorrecto
    not_string = df[~df["combined_features"].apply(lambda x: isinstance(x, str))]
    print("❌ Filas que NO son string:", len(not_string))

    # 2️⃣ Contienen sintaxis de dict/lista sin limpiar
    pattern_structure = r"[\{\}\[\]:]"
    weird_structure = df[df["combined_features"].str.contains(pattern_structure, regex=True, na=False)]
    print("❌ Filas con `{ }`, `[ ]` o `:` :", len(weird_structure))

    # 3️⃣ Dobles espacios
    double_spaces = df[df["combined_features"].str.contains(r"\s{2,}", regex=True, na=False)]
    print("⚠️ Filas con dobles espacios:", len(double_spaces))

    # 4️⃣ Texto literal problemático
    weird_text = df[df["combined_features"].str.contains(r"\b(None|nan)\b", regex=True, na=False)]
    print("❌ Filas con 'None' o 'nan' literal:", len(weird_text))

    # 5️⃣ Vacías
    empty_rows = df[df["combined_features"].str.strip() == ""]
    print("ℹ️ Filas vacías:", len(empty_rows))

    print("\n📊 RESUMEN:")
    print("Total filas:", total_rows)
    print("Porcentaje vacías:", round(len(empty_rows)/total_rows*100, 2), "%")

    print("\n🔎 Ejemplos problemáticos (si existen):")

    if len(weird_structure) > 0:
        print("\nEjemplo estructura rara:")
        print(weird_structure["combined_features"].head(3))

    if len(double_spaces) > 0:
        print("\nEjemplo dobles espacios:")
        print(double_spaces["combined_features"].head(3))

    if len(weird_text) > 0:
        print("\nEjemplo texto raro:")
        print(weird_text["combined_features"].head(3))

    print("\n✅ Auditoría finalizada.")

In [33]:
audit_combined_features(df)

🔎 INICIANDO AUDITORÍA

❌ Filas que NO son string: 0
❌ Filas con `{ }`, `[ ]` o `:` : 0
⚠️ Filas con dobles espacios: 0


C:\Users\PabloR\AppData\Local\Temp\ipykernel_60232\3603578736.py:23: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  weird_text = df[df["combined_features"].str.contains(r"\b(None|nan)\b", regex=True, na=False)]


❌ Filas con 'None' o 'nan' literal: 0
ℹ️ Filas vacías: 7956

📊 RESUMEN:
Total filas: 121455
Porcentaje vacías: 6.55 %

🔎 Ejemplos problemáticos (si existen):

✅ Auditoría finalizada.


In [34]:
import numpy as np
import pandas as pd

def build_user_vector(df, user_games, X):

    if X is None:
        return None

    # Asegurarse de que app_id sea numérico
    df["app_id"] = pd.to_numeric(df["app_id"], errors="coerce")

    user_appids = user_games["appid"].astype(int).tolist()
    indices = df[df["app_id"].isin(user_appids)].index

    if len(indices) == 0:
        return None

    # Promedio de los vectores de los juegos del usuario
    user_vector = X[indices].mean(axis=0)

    # 🔥 Convertimos a array normal
    user_vector = np.asarray(user_vector)

    return user_vector

al recomendar un juego asegurarme que no recomiende uno que el usuario ya tenga

In [35]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

def recommend(df, X, user_vector, user_games, top_n=10):

    if X is None or user_vector is None:
        print("❌ X o user_vector son None")
        return pd.DataFrame()

    # Asegurar que app_id sea numérico
    df["app_id"] = pd.to_numeric(df["app_id"], errors="coerce")

    # Calcular similitud
    similarities = cosine_similarity(user_vector, X)
    scores = similarities.flatten()

    df = df.copy()
    df["similarity"] = scores

    # 🔹 Obtener juegos del usuario
    user_appids = set(user_games["appid"].astype(int))

    # 🔹 No recomendar juegos que el usuario ya posee
    recommendations = df[~df["app_id"].isin(user_appids)]

    # Ordenar por similitud
    recommendations = recommendations.sort_values("similarity", ascending=False)

    return recommendations.head(top_n)

In [ ]:
user_games = get_user_games(76561198215235853)
user_games

2026-03-18 17:28:04,069 - INFO - Consultando juegos del usuario 76561198106607325
2026-03-18 17:28:04,070 - INFO - Enviando petición a la API de Steam
2026-03-18 17:28:04,655 - INFO - Código de respuesta: 200
2026-03-18 17:28:04,656 - INFO - Se encontraron 199 juegos


,appid,name,playtime_forever,img_icon_url,has_community_visible_stats,playtime_windows_forever,playtime_mac_forever,playtime_linux_forever,playtime_deck_forever,rtime_last_played,playtime_disconnected,content_descriptorids,has_leaderboards,playtime_2weeks
0,400,Portal,138,cfa928ab4119dd137e50d728e8fe703e4e970aff,True,0,0,0,0,1563138414,0,NaN,NaN,NaN
1,550,Left 4 Dead 2,970,7d5a243f9500d2f8467312822f8af2a2928777ed,True,327,0,0,0,1644199786,0,"[2, 5]",NaN,NaN
2,570,Dota 2,1337,0bbb630d63262dd66d2fdd0f7d37e8661a410075,NaN,0,0,0,0,1502904370,0,[5],NaN,NaN
3,620,Portal 2,384,25a5a16b2423bf7487ac5340b5b0948cef48c5f8,True,366,0,0,0,1644080230,0,NaN,True,NaN
4,730,Counter-Strike 2,23674,8dbc71957312bbd3baea65848b545be9eae2a355,True,7105,0,0,0,1696342215,0,"[2, 5]",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194,2835570,Buckshot Roulette,43,81779afdc4209795d9ccf7948514b9a227acf601,True,43,0,0,0,1730296980,0,NaN,NaN,NaN
195,3065170,Monster Hunter Wilds Beta test,109,cb8345c0f9742dad3764d058c9da49c8860d15f9,NaN,109,0,0,0,1730475956,0,NaN,NaN,NaN
196,3224770,Umamusume: Pretty Derby,864,9286b27b944818b067b70fc03d0ab88affdbe7c0,True,864,0,0,0,1759706374,0,NaN,NaN,NaN
197,3564740,Where Winds Meet,47,531ce29af277670c3aedc30d3b0bef92ee1b18c4,True,47,0,0,0,1763162870,0,NaN,NaN,NaN


In [40]:
print("Tipo appID en dataframe:", df["app_id"].dtype)

print("Tipo primer appid usuario:", type(user_games.iloc[0]["appid"]))

Tipo appID en dataframe: object
Tipo primer appid usuario: <class 'numpy.int64'>


In [42]:
build_user_profile(df, user_games)

2026-03-18 17:32:26,602 - INFO - Juegos con al menos 120 minutos jugados: 106
2026-03-18 17:32:26,603 - INFO - AppIDs válidos: 106
2026-03-18 17:32:26,620 - INFO - Juegos encontrados en dataset: 100


,app_id,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags,combined_features
45509,730,Counter-Strike 2,"Aug 21, 2012",0,0.00,1,"For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...",,...,1173003,100000000 - 200000000,33906,702,6237,307,0,1013936,"{'FPS': 91172, 'Shooter': 65634, 'Multiplayer'...",Action Free To Play FPS Shooter Multiplayer Co...
4193,570,Dota 2,"Jul 9, 2013",0,0.00,2,"The most-played game on Steam. Every day, mill...","The most-played game on Steam. Every day, mill...","Every day, millions of players worldwide enter...",“A modern multiplayer masterpiece.” 9.5/10 – D...,...,461826,100000000 - 200000000,54064,1427,1155,860,0,623941,"{'Free to Play': 60040, 'MOBA': 20225, 'Multip...",Action Strategy Free To Play to MOBA Multiplay...
8878,578080,PUBG: BATTLEGROUNDS,"Dec 21, 2017",13,0.00,0,LAND Drop into an ever-growing and changin...,LAND Drop into an ever-growing and changin...,"PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",,...,1037487,100000000 - 200000000,23787,730,6082,302,0,314682,"{'Survival': 14893, 'Shooter': 12788, 'Battle ...",Action Adventure Massively Multiplayer Free To...
5377,252490,Rust,"Feb 8, 2018",0,19.99,5,The only aim in Rust is to survive. Everything...,The only aim in Rust is to survive. Everything...,The only aim in Rust is to survive. Everything...,"“Rust is one of the cruelest games on Steam, a...",...,156649,20000000 - 50000000,24264,1246,3044,373,50,143870,"{'Survival': 18782, 'Crafting': 11939, 'Multip...",Action Adventure Indie Massively Multiplayer R...
54502,1172470,Apex Legends™,"Nov 4, 2020",0,0.00,0,Apex Legends: Amped About the Game Conquer wit...,"Conquer with character in Apex Legends, a free...","Apex Legends is the award-winning, free-to-pla...",“The champion of Battle Royales.” 9/10 – GameS...,...,326926,100000000 - 200000000,10073,615,916,269,0,124262,"{'Free to Play': 2232, 'Battle Royale': 1511, ...",Action Adventure Free To Play to Battle Royale...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65222,1623660,MIR4,"Aug 25, 2021",0,0.00,0,Venture into the open world of MIR and start y...,Venture into the open world of MIR and start y...,MIR4 is a free-to-play open world K-fantasy MM...,,...,0,0 - 0,0,0,0,0,0,0,[],Action Adventure Massively Multiplayer RPG Fre...
65261,1808500,ARC Raiders,"Oct 30, 2025",0,31.99,2,ARC Raiders Deluxe Edition ARC Raiders Deluxe ...,"SCAVENGE, SURVIVE, THRIVE In ARC Raiders, gam...",ARC Raiders is a multiplayer extraction advent...,,...,0,0 - 20000,3093,974,2007,448,20,0,[],Action Multi-player PvP Online Co-op Cross-Pla...
79231,2076040,The Finals Playtest,"Oct 26, 2023",0,0.00,0,,,,,...,0,0 - 0,0,0,0,0,0,0,[],
82425,1607250,MY HERO ULTRA RUMBLE,"Sep 28, 2023",0,0.00,0,Season 14 Begins! Season 14 sees the highly an...,Season 14 Begins! Season 14 sees the highly an...,"Pick your favorite character, team up and figh...",,...,0,0 - 0,0,0,0,0,0,0,[],Action Casual Free To Play Single-player Multi...


In [43]:
# =====================================================
# EJEMPLO DE USO
# =====================================================

steam_url = "https://steamcommunity.com/profiles/76561198367022349/"

if __name__ == "__main__":

    import pandas as pd
    
    # 1️⃣ Cargar dataset
    print("🔄 Cargando dataset...")
    
    # 2️⃣ Preparar texto y matriz TF-IDF
    print("🧠 Preparando features...")
    df = prepare_text(df)

    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer(stop_words="english")
    X = vectorizer.fit_transform(df["combined_features"])

    # 3️⃣ Pedir URL de Steam
    url_usuario = input("\nIntroduce tu perfil de Steam: ")

    steamid = extract_steamid(url_usuario)

    if not steamid:
        print("❌ No se pudo obtener el SteamID.")
        exit()

    print("✅ SteamID detectado:", steamid)

    # 4️⃣ Obtener juegos del usuario
    juegos_usuario = get_user_games(steamid)

    if juegos_usuario.empty:
        print("⚠️ Perfil privado o sin juegos.")
        exit()

    print(f"🎮 Juegos encontrados: {len(juegos_usuario)}")

    # 5️⃣ Construir vector del usuario
    user_vector = build_user_vector(df, juegos_usuario, X)

    if user_vector is None:
        print("⚠️ No se pudieron mapear los juegos al dataset.")
        exit()
    else: 
        print("Conseguido")

    # 6️⃣ Generar recomendaciones
    recomendaciones = recommend(df, X, user_vector, juegos_usuario, top_n=10)

    print("\n🎯 RECOMENDACIONES PERSONALIZADAS:\n")

    for i, row in recomendaciones.iterrows():
        print(f"🎮 {row['name']}")
        print(f"   Similaridad: {row['similarity']:.4f}")
        print(f"   Géneros: {row['genres']}")
        print(f"   Tags: {row['tags']}")
        print(f"   Categorias: {row['categories']}")


        print("-" * 50)

🔄 Cargando dataset...
🧠 Preparando features...


NameError: name 'extraer_steamid' is not defined

In [ ]:
url_usuario = "https://steamcommunity.com/profiles/76561198367022349"

print("🔄 Preparando texto...")
df = preparar_texto(df)
print("✅ combined_features creado")

print("🧠 Creando TF-IDF...")
vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(df["combined_features"])
print(f"✅ TF-IDF listo. Shape: {X.shape}")

print("\n🔗 Obteniendo SteamID...")
url_usuario = input("Introduce tu URL de Steam: ")
steamid = extraer_steamid(url_usuario)

if not steamid:
    print("❌ No se pudo obtener SteamID")
    exit()

print(f"✅ SteamID: {steamid}")

print("🎮 Obteniendo juegos del usuario...")
juegos_usuario = obtener_juegos_usuario(steamid)

if not juegos_usuario:
    print("⚠️ No se encontraron juegos o perfil privado")
    exit()

print(f"✅ Juegos encontrados: {len(juegos_usuario)}")

print("🧠 Construyendo vector del usuario...")
user_vector = construir_vector_usuario(df, juegos_usuario, X)

if user_vector is None:
    print("❌ No se pudo construir vector del usuario")
    exit()

print("✅ Vector del usuario creado")

print("🎯 Generando recomendaciones...")
recomendaciones = recomendar(df, X, user_vector, juegos_usuario)

if recomendaciones is None or recomendaciones.empty:
    print("⚠️ No se encontraron recomendaciones")
else:
    print("\n🎉 RECOMENDACIONES:")
    print(recomendaciones[["name", "similarity"]].head(10))

🔄 Preparando texto...
✅ combined_features creado
🧠 Creando TF-IDF...
✅ TF-IDF listo. Shape: (122611, 544)

🔗 Obteniendo SteamID...


2026-03-15 22:07:42,331 - INFO - SteamID numérico detectado
2026-03-15 22:07:42,333 - INFO - Consultando juegos del usuario 76561198367022349
2026-03-15 22:07:42,333 - INFO - Enviando petición a la API de Steam


✅ SteamID: 76561198367022349
🎮 Obteniendo juegos del usuario...


2026-03-15 22:07:42,630 - INFO - Código de respuesta: 200
2026-03-15 22:07:42,631 - INFO - Se encontraron 76 juegos


ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:
import numpy as np

def construir_vector_usuario(df, juegos_usuario, feature_matrix):
    
    appids_usuario = [j["appid"] for j in juegos_usuario]
    
    indices = df[df["appID"].isin(appids_usuario)].index
    
    if len(indices) == 0:
        return None
    
    # Vector promedio
    user_vector = np.asarray(feature_matrix[indices].mean(axis=0))
    
    return user_vector

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df.genres)